# Supply Chain & Inventory Analytics — Data Profiling

## Objective

This notebook profiles the raw supply-chain dataset and validates its
structure, quality, time coverage, business dimensions, and data grain.

Detailed business analysis is performed separately in:

`02_business_analysis.ipynb`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")

In [ ]:
DATA_PATH = Path("../data/raw/supply_chain_dataset.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {DATA_PATH.resolve()}"
    )

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

## 1. Dataset Overview

In [ ]:
print("Dataset Shape")
print("-" * 40)
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

print("\nColumn Names")
print("-" * 40)

for column in df.columns:
    print(column)

In [ ]:
display(df.head())

In [ ]:
display(df.tail())

In [ ]:
print("Data Types")
print("-" * 40)

display(
    df.dtypes.to_frame("Data_Type")
)

In [ ]:
print("Numerical Summary")

display(
    df.describe().T
)

In [ ]:
print("Categorical Summary")

display(
    df.describe(include="object").T
)

## 2. Data Quality Assessment

In [ ]:
quality_report = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Missing_Values": df.isna().sum().values,
    "Missing_Percentage": (
        df.isna().mean().values * 100
    ).round(2),
    "Unique_Values": df.nunique(dropna=False).values
})

display(quality_report)

In [ ]:
duplicate_count = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_count:,}")

In [ ]:
numeric_columns = df.select_dtypes(
    include=np.number
).columns

negative_report = pd.DataFrame({
    "Column": numeric_columns,
    "Negative_Value_Count": [
        (df[column] < 0).sum()
        for column in numeric_columns
    ]
})

display(negative_report)

In [ ]:
infinite_report = pd.DataFrame({
    "Column": numeric_columns,
    "Infinite_Value_Count": [
        np.isinf(df[column]).sum()
        for column in numeric_columns
    ]
})

display(infinite_report)

## 3. Date Validation

In [ ]:
df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce"
)

invalid_dates = df["Date"].isna().sum()

print(f"Invalid dates: {invalid_dates:,}")
print(f"Minimum date: {df['Date'].min()}")
print(f"Maximum date: {df['Date'].max()}")
print(f"Unique dates: {df['Date'].nunique():,}")

In [ ]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Month_Name"] = df["Date"].dt.strftime("%b")

monthly_records = (
    df.groupby(
        ["Year", "Month", "Month_Name"]
    )
    .size()
    .reset_index(name="Record_Count")
    .sort_values(["Year", "Month"])
)

display(monthly_records)

In [ ]:
daily_records = (
    df.groupby("Date")
    .size()
    .reset_index(name="Record_Count")
)

plt.figure(figsize=(12, 5))

plt.plot(
    daily_records["Date"],
    daily_records["Record_Count"]
)

plt.title("Records Over Time")
plt.xlabel("Date")
plt.ylabel("Record Count")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Business Dimensions

In [ ]:
dimension_columns = [
    "SKU_ID",
    "Warehouse_ID",
    "Supplier_ID",
    "Region"
]

dimension_summary = pd.DataFrame({
    "Dimension": dimension_columns,
    "Unique_Values": [
        df[column].nunique()
        for column in dimension_columns
    ]
})

display(dimension_summary)

In [ ]:
for column in dimension_columns:

    print(f"\n{column}")

    display(
        df[column]
        .value_counts()
        .rename_axis(column)
        .reset_index(name="Record_Count")
    )

## 5. Data Grain Validation

Expected operational grain:

`Date + SKU_ID + Warehouse_ID`

In [ ]:
business_key = [
    "Date",
    "SKU_ID",
    "Warehouse_ID"
]

total_records = len(df)

unique_business_keys = (
    df[business_key]
    .drop_duplicates()
    .shape[0]
)

duplicate_business_keys = (
    total_records - unique_business_keys
)

print(f"Total records: {total_records:,}")
print(f"Unique business keys: {unique_business_keys:,}")
print(
    f"Duplicate business-key records: "
    f"{duplicate_business_keys:,}"
)

In [ ]:
business_key_duplicates = (
    df.groupby(business_key)
    .size()
    .reset_index(name="Record_Count")
)

business_key_duplicates = (
    business_key_duplicates[
        business_key_duplicates["Record_Count"] > 1
    ]
)

display(
    business_key_duplicates.head(20)
)

## 6. Profiling Summary

In [ ]:
profiling_summary = pd.DataFrame({
    "Metric": [
        "Total Records",
        "Total Columns",
        "Unique SKUs",
        "Unique Warehouses",
        "Unique Suppliers",
        "Unique Regions",
        "Unique Dates",
        "Duplicate Rows",
        "Duplicate Business Keys",
        "Invalid Dates"
    ],
    "Value": [
        len(df),
        df.shape[1],
        df["SKU_ID"].nunique(),
        df["Warehouse_ID"].nunique(),
        df["Supplier_ID"].nunique(),
        df["Region"].nunique(),
        df["Date"].nunique(),
        duplicate_count,
        duplicate_business_keys,
        invalid_dates
    ]
})

display(profiling_summary)

## Conclusion

The dataset has been profiled for structure, data quality, temporal coverage,
business dimensions, and operational grain.

The validated dataset is used by `02_business_analysis.ipynb` for detailed
business analysis.